In [1]:

from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Cargar el archivo CSV con las características de audio y etiquetas
df = pd.read_csv('audio_features_ravdess_def.csv')

# Separar las características (X) y las etiquetas (y)
X = df.drop('label', axis=1).values
y = df['label'].values

# Codificar las etiquetas como números enteros
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Reshape de X para asegurar la forma (n_samples, n_timesteps, n_features_por_timestep)
n_samples, n_features_flat = X.shape
n_timesteps = 101  # Debe coincidir con el valor que usaste en extract_features_with_timesteps
n_features = n_features_flat // n_timesteps

X_reshaped = X.reshape(n_samples, n_timesteps, n_features)

# Convertir a tensores de PyTorch
X_tensor = torch.tensor(X_reshaped, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# Crear DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Definición del modelo LSTM para reconocimiento de emociones
class EmotionRecognitionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(EmotionRecognitionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.5)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        h_0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        c_0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h_0, c_0))
        out = out[:, -1, :]  # Tomar el último timestep
        out = self.fc(out)
        return out



# Parámetros del modelo
input_size = n_features  # Número de características por timestep
hidden_size = 64  # Número de unidades ocultas en la LSTM
num_layers = 1  # Número de capas LSTM
num_classes = len(label_encoder.classes_)  # Número de clases (emociones)

# Instanciar el modelo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmotionRecognitionLSTM(input_size, hidden_size, num_layers, num_classes).to(device)

# Definir la función de pérdida y el optimizador
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




/Users/sofiaguerrero/miniconda3/lib/python3.12/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold



# Parámetros del modelo
input_size = n_features  # Número de características por timestep
hidden_size = 256        # Número de unidades ocultas en la LSTM
num_layers = 3           # Número de capas LSTM
num_classes = len(label_encoder.classes_)  # Número de clases (emociones)


# Configurar los resultados de validación
fold_accuracies = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Entrenamiento y validación usando K-Fold Cross-Validation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for fold, (train_index, val_index) in enumerate(kf.split(X_tensor)):
    print(f'Fold {fold + 1}/{kf.get_n_splits()}')
    
    # Dividir los datos en entrenamiento y validación
    X_train, X_val = X_tensor[train_index], X_tensor[val_index]
    y_train, y_val = y_tensor[train_index], y_tensor[val_index]
    
    # Crear DataLoaders
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Instanciar el modelo
    model = EmotionRecognitionLSTM(input_size, hidden_size, num_layers, num_classes)
    model = model.to(device)
    
    # Definir función de pérdida y optimizador
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Entrenamiento
    num_epochs = 50
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}')
    
    # Validación
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * correct / total
    fold_accuracies.append(val_accuracy)
    print(f'Validation Accuracy for Fold {fold + 1}: {val_accuracy}%')

# Promedio de la precisión en todos los folds
mean_accuracy = np.mean(fold_accuracies)
print(f'Mean Cross-Validation Accuracy: {mean_accuracy}%')




Fold 1/5
Epoch [1/50], Loss: 2.0162501335144043
Epoch [2/50], Loss: 1.8043000168270535
Epoch [3/50], Loss: 1.7665365139643352
Epoch [4/50], Loss: 1.7315212388833363
Epoch [5/50], Loss: 1.7055354350143008
Epoch [6/50], Loss: 1.6622403197818332
Epoch [7/50], Loss: 1.647602962123023
Epoch [8/50], Loss: 1.5930623080995348
Epoch [9/50], Loss: 1.5967301262749567
Epoch [10/50], Loss: 1.5419528186321259
Epoch [11/50], Loss: 1.5497815940115187
Epoch [12/50], Loss: 1.5388383467992146
Epoch [13/50], Loss: 1.5233382383982341
Epoch [14/50], Loss: 1.5007885032229953
Epoch [15/50], Loss: 1.4602941109074488
Epoch [16/50], Loss: 1.4689287510183122
Epoch [17/50], Loss: 1.5697864194711049
Epoch [18/50], Loss: 1.4787492983871036
Epoch [19/50], Loss: 1.387640161646737
Epoch [20/50], Loss: 1.4351865351200104
Epoch [21/50], Loss: 1.3690107862154643
Epoch [22/50], Loss: 1.4104183846049838
Epoch [23/50], Loss: 1.3320460352632735
Epoch [24/50], Loss: 1.231720796889729
Epoch [25/50], Loss: 1.2350635512007608
Epo

In [5]:
torch.save(model.state_dict(), "final_model.pth")


Fold 1/5
Epoch [1/50], Loss: 2.0032422807481556
Epoch [2/50], Loss: 1.8498677213986714
Epoch [3/50], Loss: 1.7819568713506062
Epoch [4/50], Loss: 1.7274579008420308
Epoch [5/50], Loss: 1.7293568683995142
Epoch [6/50], Loss: 1.6801807185014088


KeyboardInterrupt: 

In [28]:
unique_classes_true = np.unique(y_true)
unique_classes_pred = np.unique(y_pred)
print(f"Clases únicas en y_true: {unique_classes_true}")
print(f"Clases únicas en y_pred: {unique_classes_pred}")


Clases únicas en y_true: []
Clases únicas en y_pred: []


In [20]:
print(f"y_true length: {len(y_true)}")
print(f"y_pred length: {len(y_pred)}")

# Si la longitud es igual a 1, puede indicar que solo estás agregando un valor escalar en lugar de una lista de valores.


with torch.no_grad():
    for features, labels in val_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)

        # Verifica que 'labels' y 'predicted' sean arrays y no escalares
        print(f"Labels shape: {labels.shape}, Predicted shape: {predicted.shape}")


y_true length: 288
y_pred length: 288
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])
Labels shape: torch.Size([32]), Predicted shape: torch.Size([32])


In [22]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Convertir a arrays NumPy si no lo son ya
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Asegúrate de que sean arrays y no escalares
print(f"y_true type: {type(y_true)}, y_pred type: {type(y_pred)}")

# Generar el reporte de clasificación
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
print(f"Matriz de confusión:\n{cm}")


# Verifica si hay valores escalares dentro de y_true o y_pred
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if isinstance(true, np.int64) or isinstance(pred, np.int64):
        print(f"Error at index {i}: y_true = {true}, y_pred = {pred}")


y_true type: <class 'numpy.ndarray'>, y_pred type: <class 'numpy.ndarray'>


TypeError: object of type 'numpy.int64' has no len()

In [24]:
# Inicializar listas vacías para etiquetas verdaderas y predicciones
y_true = []
y_pred = []

# Colocar el modelo en modo evaluación
model.eval()
with torch.no_grad():
    for features, labels in val_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        
        # Acumular las etiquetas verdaderas y las predicciones
        y_true.extend(labels.cpu().numpy())  # Asegurarse de agregar múltiples etiquetas
        y_pred.extend(predicted.cpu().numpy())  # Asegurarse de agregar múltiples predicciones

# Después de recolectar todas las predicciones y etiquetas verdaderas,
# pasarlas a las funciones de evaluación

# Reporte de clasificación
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# Matriz de confusión
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Matriz de Confusión')
plt.show()


TypeError: object of type 'numpy.int64' has no len()